# Early vs. Late Layer at Matched Relative Push -- Mistral-Prot-134M

## The bug being fixed

The original early-vs-late layer experiments (`notes/locked-results.md` §1g/§1h/§1i/§1k) pushed both
layers with a steering vector rescaled to the same **absolute** norm, `REFERENCE_NORM = 583.998`.

That is not a fair comparison. The residual stream **accumulates with depth** -- each layer adds to a
running total -- so a late layer's hidden state is naturally larger than an early layer's. Injecting
the same absolute vector therefore displaces the early layer's hidden state by a **larger fraction**.
`34-ai4dd-residual-norm-audit.ipynb` measured this directly and found it in **5 of 5 models**, with
the early layer over-pushed by 1.28x to 4.47x. That is precisely the shape of the reported
"early layers are more sensitive" trend, whether or not early layers are actually more fragile.

Note the confound is **not** about hidden dimension. ProtGPT2 and ZymCTRL have identical
1280-dim/36-layer shapes and still differ ~130x in residual-stream norm. What matters is measured
‖h‖, not architecture size.

## The fix

Scale each layer's vector to the same **alpha_rel = ‖v_inject‖ / ‖h at that layer‖**, so both layers
are displaced by the same fraction of their own hidden state. The anchor is ProtGPT2's own layer-12
alpha_rel (~0.200) -- the same anchor used by `38`/`40`/`41`/`47`, so this result is directly
comparable to the rest of the repaired picture.

`41` already did this for p-IgGen (result: 0.0% collapse at both layers, both doses -- the gap
vanished entirely). This notebook does it for Mistral-Prot-134M.

## Why this dose ladder

**Run this one last, and expect it to be uninformative.** Mistral-Prot has a 99.0% natural collapse rate and a pool mean pLDDT of 40.78 (§1j) -- there is essentially no headroom for steering to make its output measurably worse on either readout. It is included for completeness of the model sweep, not because a result is expected.

It does have one thing going for it: at 4.47x it had the **largest** early/late push asymmetry of any model in 34's audit, so it was the most confounded. Three doses are run (1x/2x/4x) rather than two, because this model has never been meaningfully pushed at all -- §1k-FLAG found its original 'steering' was alpha_rel 0.002-0.021, i.e. a 0.2% displacement, indistinguishable from no intervention.

If the responsiveness check at the end reports INERT, log this as 'dose too low / no headroom', **not** as 'no layer effect'.

## What a null means here -- read this before logging the result

A null is only interpretable if the model **responded to steering at all**. Two inert conditions
look identical for reasons that have nothing to do with layer depth. The final cell therefore runs
an explicit **responsiveness check** against control, and the verdict distinguishes:

- *inert* -> the dose was too low to test the layer question. Not a layer result.
- *responded, no layer difference* -> a real, interpretable negative.
- *responded, layer difference survives* -> the first properly-controlled positive in the project.

## Readouts

Both, deliberately: **collapse rate** (Fisher exact) for comparability with earlier notebooks, and
**mean pLDDT** (Mann-Whitney U) as the more sensitive continuous measure. §1k-REPAIRED is the
precedent -- on a model whose pool already collapses ~85 percent of the time, the binary rate is ceilinged
and only pLDDT could show that RITA genuinely ignored the intervention.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect ~2-2.5 hours (7 conditions, 350 folds).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc
import math
import collections
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from scipy.stats import fisher_exact, mannwhitneyu

torch.manual_seed(2026)
np.random.seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ProtGPT2's own natural v_L norm from notebook 03. Used ONLY to define the anchor and to report
# how far the old absolute-norm runs were from matched -- never to scale anything in this notebook.
REFERENCE_NORM = 583.998

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def calculate_entropy(seq_str):
    if not seq_str:
        return 0.0
    counts = collections.Counter(seq_str)
    total = len(seq_str)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

print("Setup complete. CUDA available:", torch.cuda.is_available())
print("Model under test: RaphaelMourad/Mistral-Prot-v1-134M")


Setup complete. CUDA available: True
Model under test: RaphaelMourad/Mistral-Prot-v1-134M


In [2]:
# --- The same common probe set as 34/38/40/41/47: real UniProt fragments. Using one shared probe
#     set across every repair notebook is what makes their alpha_rel values directly comparable
#     rather than each notebook being its own island. ---

UNIPROT_ACCESSIONS = [
    "P0CG48", "P00720", "P02144", "P42212", "P01308", "P61823",
    "P00648", "P99999", "P69905", "P68871", "P00698", "P00441",
]

def fetch_uniprot_sequence(accession, timeout=10):
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            text = resp.read().decode("utf-8")
        lines = [l for l in text.strip().split("\n") if l]
        seq = "".join(lines[1:])
        return seq if len(seq) >= 20 else None
    except Exception as e:
        print(f"  skip {accession}: {e}")
        return None

FALLBACK_SEQS = [
    "NLYIQWLKDGGPSSGRPPPS",
    "LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF",
    "GSQIGAKNTGQVQLNLLAL",
    "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
    "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
    "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV",
]

print("Fetching real reference protein sequences from UniProt...")
reference_seqs = []
for acc in UNIPROT_ACCESSIONS:
    seq = fetch_uniprot_sequence(acc)
    if seq:
        reference_seqs.append((acc, seq))
        print(f"  fetched {acc}: {len(seq)} residues")

USED_FALLBACK = False
if not reference_seqs:
    USED_FALLBACK = True
    print("!" * 78)
    print("!! UniProt fetch returned NOTHING -- Kaggle's Internet toggle is likely OFF.")
    print("!! Fix: Notebook sidebar -> Session options -> Internet -> ON, then re-run.")
    print("!! Falling back to hardcoded reference sequences so the run can proceed, but the probe")
    print("!! set will then differ from 34/38/40/41/47 -- say so if you log this result.")
    print("!" * 78)
    reference_seqs = [(f"local{i + 1}", s) for i, s in enumerate(FALLBACK_SEQS)]

def build_probe_set(reference_seqs, n_probes=40, frag_len=50, seed=7):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    probes = []
    for i in range(n_probes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        L = min(frag_len, len(seq))
        start = rng.randint(0, max(1, len(seq) - L + 1))
        probes.append(seq[start:start + L])
    return probes

def build_prefix_pool(reference_seqs, n_prefixes, min_len=10, max_len=15, seed=11):
    if not reference_seqs:
        raise RuntimeError("No reference sequences available -- check Internet settings.")
    rng = np.random.RandomState(seed)
    prefixes = []
    for i in range(n_prefixes):
        acc, seq = reference_seqs[i % len(reference_seqs)]
        plen = rng.randint(min_len, max_len + 1)
        start = rng.randint(0, max(1, len(seq) - plen))
        prefixes.append(seq[start:start + plen])
    return prefixes

probe_seqs = build_probe_set(reference_seqs, n_probes=40)
print(f"\nBuilt {len(probe_seqs)} common probe fragments from {len(reference_seqs)} source proteins.")

def get_layer_module(model, path, idx):
    obj = model
    for part in path.split("."):
        obj = getattr(obj, part)
    return obj[idx]

def measure_resid_norm_mod(model, layer_module, tokenize_fn, seqs):
    # Mean per-position residual-stream norm at one layer. This is the ‖h‖ that alpha_rel
    # divides by, and measuring it per-layer is the entire point of this notebook.
    captured = {}
    def hook(module, inp, out):
        h = out[0] if isinstance(out, (tuple, list)) else out
        captured["h"] = h.detach()
    handle = layer_module.register_forward_hook(hook)
    vals = []
    try:
        for seq in seqs:
            enc = tokenize_fn(seq)
            captured.clear()
            with torch.no_grad():
                model(**enc)
            if "h" in captured:
                vals.append(captured["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

print("Probe set and residual-norm measurement ready.")


Fetching real reference protein sequences from UniProt...
  fetched P0CG48: 685 residues
  fetched P00720: 164 residues
  fetched P02144: 154 residues
  fetched P42212: 238 residues
  fetched P01308: 110 residues
  fetched P61823: 150 residues
  fetched P00648: 157 residues
  fetched P99999: 105 residues
  fetched P69905: 142 residues
  fetched P68871: 147 residues
  fetched P00698: 147 residues
  fetched P00441: 154 residues

Built 40 common probe fragments from 12 source proteins.
Probe set and residual-norm measurement ready.


In [3]:
# --- The anchor: ProtGPT2's own layer-12 alpha_rel, the SAME anchor 38/40/41/47 used. Every dose
#     in this notebook is a multiple of this number, so the result sits on the same scale as the
#     rest of the repaired cross-model picture instead of being internally-consistent-only. ---

print(f"Loading ProtGPT2 on {device} to measure the anchor...")
anchor_tokenizer = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
anchor_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device)
anchor_model.eval()

def anchor_tok(s):
    return anchor_tokenizer(s, return_tensors="pt", truncation=True, max_length=256).to(device)

h_protgpt2_l12 = measure_resid_norm_mod(
    anchor_model, get_layer_module(anchor_model, "transformer.h", 12), anchor_tok, probe_seqs)
anchor_alpha_rel = REFERENCE_NORM / h_protgpt2_l12

print(f"ProtGPT2 layer 12 mean residual-stream norm (this run's probes): {h_protgpt2_l12:.2f}")
print(f"anchor_alpha_rel = {anchor_alpha_rel:.4f}")
print("(34 measured 2921.53 -> 0.2000; 38/40/41/47 each reproduced ~0.2000. A close value here")
print(" confirms the anchor is a stable measured quantity rather than a fluke of one run.)")

del anchor_model, anchor_tokenizer
clear_gpu()
print("\nProtGPT2 anchor freed from GPU.")


Loading ProtGPT2 on cuda to measure the anchor...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtGPT2 layer 12 mean residual-stream norm (this run's probes): 2919.92
anchor_alpha_rel = 0.2000
(34 measured 2921.53 -> 0.2000; 38/40/41/47 each reproduced ~0.2000. A close value here
 confirms the anchor is a stable measured quantity rather than a fluke of one run.)

ProtGPT2 anchor freed from GPU.


In [4]:
# --- Scoring + ESMFold. Identical to 24/26/27/41, including the length-aware OOM retry added in
#     39/40 and the fold_ok tracking added after the §1l collapse-metric hole. ---
ALPHABET_SIZE = 20

def h_norm(seq):
    if not seq:
        return 0.0
    counts = collections.Counter(seq)
    total = len(seq)
    ent = -sum((c / total) * math.log2(c / total) for c in counts.values())
    return ent / math.log2(ALPHABET_SIZE)

def distinct_n(seq, n):
    if len(seq) < n:
        return 1.0
    grams = [seq[i:i + n] for i in range(len(seq) - n + 1)]
    return len(set(grams)) / len(grams)

def homopolymer_runs(seq):
    if not seq:
        return []
    runs, run_len = [], 1
    for i in range(1, len(seq)):
        if seq[i] == seq[i - 1]:
            run_len += 1
        else:
            runs.append(run_len)
            run_len = 1
    runs.append(run_len)
    return runs

def r_hpoly(seq, k=4):
    if not seq:
        return 1.0
    T = len(seq)
    penalty = sum(l for l in homopolymer_runs(seq) if l >= k)
    return max(0.0, 1.0 - penalty / T)

def repetition_score(seq):
    return float(np.mean([h_norm(seq), distinct_n(seq, 2), distinct_n(seq, 3), r_hpoly(seq)]))

def utility_score(plddt, ptm):
    return float(np.mean([plddt / 100.0, ptm]))

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

class StructuralEvaluatorPTM:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading ESMFold...")
        self.tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
        self.model = self.model.to(self.device).eval()

    def fold_one(self, seq, max_len=300):
        cleaned = "".join(a for a in seq if a in VALID_AA)
        if len(cleaned) < 10:
            return 0.0, 0.0, False
        working = cleaned[:max_len] if len(cleaned) > max_len else cleaned
        try:
            inputs = self.tokenizer([working], return_tensors="pt", add_special_tokens=False).to(self.device)
            with torch.no_grad():
                out = self.model(**inputs)
            raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
            plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
            ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
            return plddt, ptm, True
        except RuntimeError:
            clear_gpu()
        half = max(10, len(working) // 2)
        if half < len(working):
            try:
                inputs = self.tokenizer([working[:half]], return_tensors="pt", add_special_tokens=False).to(self.device)
                with torch.no_grad():
                    out = self.model(**inputs)
                raw_plddt = float(np.mean(out.plddt.cpu().numpy()))
                plddt = raw_plddt * 100.0 if raw_plddt <= 1.5 else raw_plddt
                ptm = float(out.ptm.item()) if hasattr(out, "ptm") else 0.0
                return plddt, ptm, True
            except RuntimeError:
                clear_gpu()
        return 0.0, 0.0, False

def fold_records_ptm(records, evaluator):
    for r in records:
        plddt, ptm, fold_ok = evaluator.fold_one(r["sequence"])
        r["plddt"] = plddt
        r["ptm"] = ptm
        r["fold_ok"] = fold_ok
        r["collapse"] = int(0.0 < plddt < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(plddt, ptm) if plddt > 0 else 0.0
    return records

print("Scoring functions and length-aware ESMFold evaluator ready.")


Scoring functions and length-aware ESMFold evaluator ready.


In [5]:
# --- Mistral-Prot-134M. READ THE HEADER BEFORE INTERPRETING ANYTHING FROM THIS NOTEBOOK.
#
#     §1j recorded a 99.0% natural collapse rate and mean pool pLDDT of 40.78 for this model. That
#     is a ceiling: there is almost no room for steering to make the output worse on the binary
#     collapse readout, and little on pLDDT either. This notebook is run for completeness of the
#     model sweep, and mean pLDDT is reported as primary, but a null here is WEAK evidence -- it
#     is consistent with "no layer effect" and equally with "no measurable range". The
#     responsiveness check at the end is what tells the two apart. ---

HOOK_PATH = "model.layers"

print(f"Loading Mistral-Prot-134M on {device}...")
tokenizer = AutoTokenizer.from_pretrained("RaphaelMourad/Mistral-Prot-v1-134M", trust_remote_code=True)
plm_model = AutoModelForCausalLM.from_pretrained(
    "RaphaelMourad/Mistral-Prot-v1-134M", trust_remote_code=True).to(device)
plm_model.eval()

N_LAYERS = plm_model.config.num_hidden_layers
LAYER_EARLY = max(1, round(N_LAYERS / 4))
LAYER_LATE = N_LAYERS - 1
print(f"Mistral-Prot has {N_LAYERS} layers -- using {LAYER_EARLY} (~1/4 depth) and {LAYER_LATE} "
      f"(final), same relative depths as §1j.")

PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

def model_tok(s):
    return tokenizer(s, return_tensors="pt", truncation=True, max_length=256).to(device)

def generate_batch(prompts, layer, vec, seed):
    torch.manual_seed(seed)
    v = None if vec is None else vec.to(device)
    def hook(module, inp, out):
        if v is None:
            return out
        # Cast to the hidden state's own dtype at injection time -- Mistral-Prot's hidden
        # states come out as bfloat16 while v was built in float32 upstream (measure_resid_norm_mod
        # and mean_activation_hooked both call .float()). Adding float32 + bfloat16 silently
        # upcasts the residual stream to float32, which then breaks the NEXT layer's matmul
        # against its bfloat16 weights ("mat1 and mat2 ... float != c10::BFloat16"). Casting v
        # to h.dtype here keeps the residual stream's dtype unchanged regardless of what dtype
        # the model happens to be loaded in.
        if isinstance(out, tuple):
            h = out[0]
            return (h + v.to(dtype=h.dtype),) + out[1:]
        return out + v.to(dtype=out.dtype)
    recs = []
    lm = get_layer_module(plm_model, HOOK_PATH, layer)
    for prompt in prompts:
        handle = lm.register_forward_hook(hook)
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = plm_model.generate(**inputs, max_length=50, do_sample=True,
                                         temperature=1.2, pad_token_id=PAD_ID)
        handle.remove()
        seq = tokenizer.decode(out_ids[0], skip_special_tokens=True).replace(" ", "")
        gen_only = seq[len(prompt):] if seq.startswith(prompt) else seq
        recs.append({"prompt": prompt, "sequence": seq, "gen_only": gen_only,
                     "entropy": calculate_entropy(gen_only)})
    clear_gpu()
    return recs

N_CANDIDATES = 200
candidate_prefixes = build_prefix_pool(reference_seqs, n_prefixes=N_CANDIDATES)
print(f"\n=== Generating N={N_CANDIDATES} natural candidates ===")
candidate_records = generate_batch(candidate_prefixes, LAYER_EARLY, None, seed=505)

print("=== Freeing Mistral-Prot while ESMFold folds the pool ===")
del plm_model
clear_gpu()

evaluator = StructuralEvaluatorPTM()
print("Folding and scoring the candidate pool...")
candidate_records = fold_records_ptm(candidate_records, evaluator)
del evaluator
clear_gpu()

valid_candidates = [r for r in candidate_records if r["plddt"] > 0.0]
pool_collapse = float(np.mean([r["collapse"] for r in valid_candidates]))
pool_plddt = float(np.mean([r["plddt"] for r in valid_candidates]))
print(f"\n{len(valid_candidates)}/{len(candidate_records)} candidates folded.")
print(f"Mean pLDDT {pool_plddt:.2f}, natural collapse {pool_collapse:.1%}")
print("(§1j recorded 99.0% natural collapse, mean pLDDT 40.78.)")
if pool_collapse > 0.90:
    print()
    print("!! CEILING CONFIRMED: the natural pool already collapses >90% of the time.")
    print("!! The binary collapse readout cannot show damage from here. Treat mean pLDDT as")
    print("!! primary, and treat any null as uninformative unless the responsiveness check")
    print("!! at the end shows the model moved at all.")


Loading Mistral-Prot-134M on cuda...


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding and scoring the candidate pool...

200/200 candidates folded.
Mean pLDDT 40.78, natural collapse 99.0%
(§1j recorded 99.0% natural collapse, mean pLDDT 40.78.)

!! CEILING CONFIRMED: the natural pool already collapses >90% of the time.
!! The binary collapse readout cannot show damage from here. Treat mean pLDDT as
!! primary, and treat any null as uninformative unless the responsiveness check
!! at the end shows the model moved at all.


In [6]:
# --- Utility-matching, unchanged from 24/26/27/41/47. Trims the extreme-repetition tails until
#     D+ and D- have comparable structural utility, so the resulting difference vector encodes
#     repetition rather than "good protein vs. junk". ---

QUANTILE = 0.30
UTILITY_TOLERANCE = 0.05

sorted_by_rep = sorted(valid_candidates, key=lambda r: r["repetition_score"])
n_side = max(10, int(len(sorted_by_rep) * QUANTILE))
d_minus_raw = sorted_by_rep[:n_side]
d_plus_raw = sorted_by_rep[-n_side:]

def utility_match(pool_a, pool_b, tolerance, max_iters=200):
    a, b = list(pool_a), list(pool_b)
    for _ in range(max_iters):
        mean_a = np.mean([r["utility_score"] for r in a])
        mean_b = np.mean([r["utility_score"] for r in b])
        gap = mean_a - mean_b
        if abs(gap) <= tolerance or min(len(a), len(b)) <= 15:
            break
        if gap > 0:
            a.sort(key=lambda r: -r["utility_score"]); a.pop(0)
        else:
            b.sort(key=lambda r: r["utility_score"]); b.pop(0)
    return a, b

d_plus, d_minus = utility_match(d_plus_raw, d_minus_raw, UTILITY_TOLERANCE)
util_gap = abs(np.mean([r["utility_score"] for r in d_plus]) - np.mean([r["utility_score"] for r in d_minus]))
rep_gap = abs(np.mean([r["repetition_score"] for r in d_plus]) - np.mean([r["repetition_score"] for r in d_minus]))
print(f"After utility-matching: D+ n={len(d_plus)}, D- n={len(d_minus)}")
print(f"  utility gap {util_gap:.3f} (target <= {UTILITY_TOLERANCE})")
print(f"  repetition separation {rep_gap:.3f} (this is the signal the vector should encode)")


After utility-matching: D+ n=60, D- n=60
  utility gap 0.006 (target <= 0.05)
  repetition separation 0.095 (this is the signal the vector should encode)


In [7]:
# --- THE ACTUAL REPAIR. ---
#
# The original fair-test notebooks scaled BOTH layers' steering vectors to the same absolute norm
# (REFERENCE_NORM = 583.998). But the residual stream grows with depth -- every layer adds to a
# running total -- so an identical absolute push displaces the EARLY layer's hidden state by a
# larger fraction. 34's audit found this in 5 of 5 models. That is exactly the shape of the
# reported "early layers are more sensitive" finding, whether or not early layers are fragile.
#
# Here each layer instead gets its own matched norm = anchor_alpha_rel * ‖h at that layer‖, so
# both layers are displaced by the SAME fraction of their own hidden state.

print(f"Reloading Mistral-Prot-134M to extract activations and measure ‖h‖ per layer...")
plm_model = AutoModelForCausalLM.from_pretrained(
    "RaphaelMourad/Mistral-Prot-v1-134M", trust_remote_code=True).to(device)
plm_model.eval()

def mean_activation_hooked(seqs, layer_module):
    cap, acts = {}, []
    def hook(m, i, o):
        cap["h"] = (o[0] if isinstance(o, (tuple, list)) else o).detach()
    handle = layer_module.register_forward_hook(hook)
    try:
        for s in seqs:
            enc = model_tok(s)
            cap.clear()
            with torch.no_grad():
                plm_model(**enc)
            if "h" in cap:
                acts.append(cap["h"].float().mean(dim=1).squeeze(0).cpu())
    finally:
        handle.remove()
    return torch.stack(acts)

d_plus_seqs = [r["sequence"] for r in d_plus]
d_minus_seqs = [r["sequence"] for r in d_minus]

steering_vectors, h_model, matched_norms, raw_norms = {}, {}, {}, {}
for layer in [LAYER_EARLY, LAYER_LATE]:
    lm = get_layer_module(plm_model, HOOK_PATH, layer)

    pos = mean_activation_hooked(d_plus_seqs, lm)
    neg = mean_activation_hooked(d_minus_seqs, lm)
    v_raw = pos.mean(dim=0) - neg.mean(dim=0)
    raw_norm = v_raw.norm().item()
    raw_norms[layer] = raw_norm

    h = measure_resid_norm_mod(plm_model, lm, model_tok, probe_seqs)
    h_model[layer] = h
    matched_norm = anchor_alpha_rel * h
    matched_norms[layer] = matched_norm

    steering_vectors[layer] = (v_raw * (matched_norm / raw_norm)).to(device)

    print(f"\nLayer {layer}:")
    print(f"  raw utility-matched ‖v_L‖   = {raw_norm:.4f}")
    print(f"  ‖h‖ at this layer            = {h:.2f}")
    print(f"  matched norm for alpha_rel={anchor_alpha_rel:.4f}  ->  {matched_norm:.4f}")
    print(f"  the original run used {REFERENCE_NORM:.3f} here regardless of layer")
    print(f"    = {REFERENCE_NORM / matched_norm:.1f}x HARDER than matched"
          f"  (alpha_rel {REFERENCE_NORM / h:.3f} instead of {anchor_alpha_rel:.3f})")

print()
print("SANITY CHECK -- matched alpha_rel must now be IDENTICAL at both layers:")
ar_e = matched_norms[LAYER_EARLY] / h_model[LAYER_EARLY]
ar_l = matched_norms[LAYER_LATE] / h_model[LAYER_LATE]
print(f"  layer {LAYER_EARLY}: {ar_e:.6f}")
print(f"  layer {LAYER_LATE}: {ar_l:.6f}")
print(f"  anchor:   {anchor_alpha_rel:.6f}")
assert abs(ar_e - anchor_alpha_rel) < 1e-6 and abs(ar_l - anchor_alpha_rel) < 1e-6, \
    "alpha_rel scaling is wrong -- do not trust anything downstream of this cell"
print("  OK: both layers are now pushed by the same fraction of their own hidden state.")

print()
print(f"For the record, the within-model push asymmetry the ORIGINAL run had:")
print(f"  {(REFERENCE_NORM / h_model[LAYER_EARLY]) / (REFERENCE_NORM / h_model[LAYER_LATE]):.2f}x"
      f"  (34 measured 4.47x -- the largest of any model)")


Reloading Mistral-Prot-134M to extract activations and measure ‖h‖ per layer...


Loading weights:   0%|          | 0/75 [00:00<?, ?it/s]


Layer 2:
  raw utility-matched ‖v_L‖   = 3541.1943
  ‖h‖ at this layer            = 57082.49
  matched norm for alpha_rel=0.2000  ->  11416.7678
  the original run used 583.998 here regardless of layer
    = 0.1x HARDER than matched  (alpha_rel 0.010 instead of 0.200)

Layer 7:
  raw utility-matched ‖v_L‖   = 18578.9102
  ‖h‖ at this layer            = 250729.40
  matched norm for alpha_rel=0.2000  ->  50147.0688
  the original run used 583.998 here regardless of layer
    = 0.0x HARDER than matched  (alpha_rel 0.002 instead of 0.200)

SANITY CHECK -- matched alpha_rel must now be IDENTICAL at both layers:
  layer 2: 0.200005
  layer 7: 0.200005
  anchor:   0.200005
  OK: both layers are now pushed by the same fraction of their own hidden state.

For the record, the within-model push asymmetry the ORIGINAL run had:
  4.39x  (34 measured 4.47x -- the largest of any model)


In [8]:
# --- Generate every condition: control, plus each layer at each dose. Doses are multiples of the
#     anchor alpha_rel, chosen for THIS model from its own measured threshold -- see the header. ---

DOSES = [1.0, 2.0, 4.0]
N_PER_CONDITION = 50

print(f"Dose ladder (multiples of anchor alpha_rel = {anchor_alpha_rel:.4f}):")
for d in DOSES:
    print(f"  {d:g}x  ->  alpha_rel = {d * anchor_alpha_rel:.3f}")
print()

conditions = {}

print("=== CONTROL (unsteered) ===")
conditions["CONTROL"] = generate_batch(build_prefix_pool(reference_seqs, N_PER_CONDITION, seed=900),
                                LAYER_EARLY, None, seed=111)

seed = 1000
for layer in [LAYER_EARLY, LAYER_LATE]:
    for dose in DOSES:
        name = f"L{layer}_D{dose:g}x"
        seed += 111
        print(f"=== {name}  (alpha_rel = {dose * anchor_alpha_rel:.3f}) ===")
        conditions[name] = generate_batch(
            build_prefix_pool(reference_seqs, N_PER_CONDITION, seed=900 + seed),
            layer, steering_vectors[layer] * dose, seed=seed)

for name, recs in conditions.items():
    print(f"{name}: {len(recs)} sequences")

print("=== Freeing Mistral-Prot-134M from GPU before ESMFold ===")
del plm_model
clear_gpu()


Dose ladder (multiples of anchor alpha_rel = 0.2000):
  1x  ->  alpha_rel = 0.200
  2x  ->  alpha_rel = 0.400
  4x  ->  alpha_rel = 0.800

=== CONTROL (unsteered) ===
=== L2_D1x  (alpha_rel = 0.200) ===
=== L2_D2x  (alpha_rel = 0.400) ===
=== L2_D4x  (alpha_rel = 0.800) ===
=== L7_D1x  (alpha_rel = 0.200) ===
=== L7_D2x  (alpha_rel = 0.400) ===
=== L7_D4x  (alpha_rel = 0.800) ===
CONTROL: 50 sequences
L2_D1x: 50 sequences
L2_D2x: 50 sequences
L2_D4x: 50 sequences
L7_D1x: 50 sequences
L7_D2x: 50 sequences
L7_D4x: 50 sequences
=== Freeing Mistral-Prot-134M from GPU before ESMFold ===


In [9]:
# --- Fold every condition, tracking fold-success separately from collapse. The §1l lesson:
#     collapse = int(0.0 < plddt < 60.0) scores a TOTAL fold failure (plddt == 0.0) as
#     NOT-collapsed, i.e. as a success. Always read fold_ok alongside the collapse rate. ---

evaluator2 = StructuralEvaluatorPTM()
for name in conditions:
    print(f"Folding {name}...")
    conditions[name] = fold_records_ptm(conditions[name], evaluator2)
del evaluator2
clear_gpu()

def wilson_ci(k, n, z=1.959963985):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = (z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / d
    return (max(0.0, centre - half), min(1.0, centre + half))

print()
hdr = "{:26s} {:>4s} {:>8s} {:>9s} {:>8s} {:>10s}".format(
    "Condition", "N", "FoldOK", "Entropy", "pLDDT", "Collapse%")
print(hdr)
print("-" * len(hdr))
summary = {}
for name, recs in conditions.items():
    n = len(recs)
    n_ok = sum(1 for r in recs if r["fold_ok"])
    k = int(np.sum([r["collapse"] for r in recs]))
    plddts = [r["plddt"] for r in recs if r["plddt"] > 0]
    summary[name] = {
        "k": k, "n": n, "rate": k / n, "ci": wilson_ci(k, n), "fold_ok": n_ok,
        "plddt_mean": float(np.mean(plddts)) if plddts else 0.0,
        "plddts": plddts,
        "entropy": float(np.mean([r["entropy"] for r in recs])),
    }
    s = summary[name]
    okstr = f"{n_ok}/{n}"
    print("{:26s} {:4d} {:>8s} {:9.3f} {:8.2f} {:9.1f}%".format(
        name, n, okstr, s["entropy"], s["plddt_mean"], s["rate"] * 100))

if any(s["fold_ok"] < s["n"] for s in summary.values()):
    print()
    print("!! Some sequences failed to fold entirely (plddt == 0.0). Those are counted as")
    print("!! NOT-collapsed by the metric. Read the FoldOK column before the collapse rates.")


Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.bias   | MISSING    | 
esm.contact_head.regression.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding CONTROL...
Folding L2_D1x...
Folding L2_D2x...
Folding L2_D4x...
Folding L7_D1x...
Folding L7_D2x...
Folding L7_D4x...

Condition                     N   FoldOK   Entropy    pLDDT  Collapse%
----------------------------------------------------------------------
CONTROL                      50    50/50     3.499    42.19      94.0%
L2_D1x                       50    50/50     3.082    52.15      70.0%
L2_D2x                       50    50/50     2.023    62.41      34.0%
L2_D4x                       50    50/50     0.377    72.76       0.0%
L7_D1x                       50    50/50     3.219    51.01      78.0%
L7_D2x                       50    50/50     3.030    53.89      74.0%
L7_D4x                       50    50/50     2.785    59.10      60.0%


In [10]:
# --- The comparison this notebook exists to make. Two readouts, deliberately:
#
#     collapse rate (binary, Fisher exact)  -- comparable to every earlier notebook
#     mean pLDDT    (continuous, Mann-Whitney U) -- PRIMARY where baseline collapse is high
#
#     §1k-REPAIRED is why both are here: on a model whose pool already collapses ~85% of the
#     time, the binary rate is ceilinged and a null on it is ambiguous, while pLDDT still has
#     20+ points of room. Reporting only the binary rate would have made that null unreadable.
# ---

BAR = "=" * 100
print(BAR)
print("EARLY vs LATE LAYER at genuinely matched relative push -- Mistral-Prot-134M")
print(BAR)
print(f"anchor_alpha_rel = {anchor_alpha_rel:.4f}   (1x dose)")
print(f"early layer = {LAYER_EARLY}, late layer = {LAYER_LATE}, of {N_LAYERS} total")
print()

print("{:26s} {:>11s} {:>8s} {:>20s} {:>9s}".format("Condition", "collapsed", "rate", "95% CI", "pLDDT"))
print("-" * 78)
for name, s in summary.items():
    lo, hi = s["ci"]
    ci_str = "[{:.1%}, {:.1%}]".format(lo, hi)
    print("{:26s} {:5d}/{:<5d} {:7.1%} {:>20s} {:9.2f}".format(
        name, s["k"], s["n"], s["rate"], ci_str, s["plddt_mean"]))

print()
print("What the ORIGINAL absolute-norm run actually pushed (the confound being repaired):")
old_early = REFERENCE_NORM / h_model[LAYER_EARLY]
old_late = REFERENCE_NORM / h_model[LAYER_LATE]
print(f"  layer {LAYER_EARLY} '1x' was really alpha_rel = {REFERENCE_NORM:.1f}/{h_model[LAYER_EARLY]:.2f} = {old_early:.3f}")
print(f"  layer {LAYER_LATE} '1x' was really alpha_rel = {REFERENCE_NORM:.1f}/{h_model[LAYER_LATE]:.2f} = {old_late:.3f}")
print(f"  -> the early layer was over-pushed by {old_early / old_late:.2f}x relative to the late layer,")
print(f"     in exactly the direction of the reported early > late trend.")
print(f"  (34's audit measured this asymmetry as 4.47x -- the largest of any model for this model.)")

print()
print(BAR)
print("VERDICT -- does an early-vs-late difference exist once the push is matched?")
print(BAR)

# Multiplicity matters here and is easy to get wrong. Each comparison below is run on TWO
# readouts, across several doses, so an uncorrected alpha=0.05 produces a false "significant"
# roughly a third of the time on 8 tests -- which would let a genuinely inert model be logged as
# "responded, no layer effect", the single most misleading outcome this notebook can produce.
# Holm-Bonferroni is applied within each family of tests, and a minimum effect size is required
# on top of it, because at N=50 a statistically significant 1-point pLDDT shift is still noise.

ALPHA = 0.05
MIN_PLDDT_SHIFT = 3.0    # points of mean pLDDT
MIN_RATE_SHIFT = 0.10    # 10 percentage points of collapse rate

def holm(pvals, alpha=ALPHA):
    # Returns a list of booleans, one per input p-value, in the ORIGINAL order.
    idx = [i for i, p in enumerate(pvals) if not (p is None or math.isnan(p))]
    ordered = sorted(idx, key=lambda i: pvals[i])
    out = [False] * len(pvals)
    m = len(ordered)
    still_rejecting = True
    for rank, i in enumerate(ordered):
        if still_rejecting and pvals[i] <= alpha / (m - rank):
            out[i] = True
        else:
            still_rejecting = False
    return out

def compare(a, b):
    # (p_collapse, p_plddt, rate_shift, plddt_shift). Shifts are SIGNED (a minus b) so the
    # direction is readable; the effect-size floors below compare against abs().
    _, p_c = fisher_exact([[a["k"], a["n"] - a["k"]], [b["k"], b["n"] - b["k"]]])
    if len(a["plddts"]) >= 3 and len(b["plddts"]) >= 3:
        _, p_p = mannwhitneyu(a["plddts"], b["plddts"], alternative="two-sided")
    else:
        p_p = float("nan")
    return p_c, p_p, a["rate"] - b["rate"], a["plddt_mean"] - b["plddt_mean"]

# --- family 1: early vs late, at each dose ---
raw = []
for dose in DOSES:
    e = summary[f"L{LAYER_EARLY}_D{dose:g}x"]
    l = summary[f"L{LAYER_LATE}_D{dose:g}x"]
    raw.append((dose, e, l) + compare(e, l))

layer_ps = [x[3] for x in raw] + [x[4] for x in raw]
layer_hits = holm(layer_ps)
n_d = len(raw)

verdict_rows = []
any_sig = False
print(f"(Holm-Bonferroni over {len([p for p in layer_ps if not math.isnan(p)])} tests; "
      f"effect floor {MIN_PLDDT_SHIFT:g} pLDDT points or {MIN_RATE_SHIFT:.0%} collapse rate)")
for i, (dose, e, l, p_c, p_p, d_rate, d_plddt) in enumerate(raw):
    sig_c = layer_hits[i] and abs(d_rate) >= MIN_RATE_SHIFT
    sig_p = layer_hits[i + n_d] and abs(d_plddt) >= MIN_PLDDT_SHIFT
    sig = sig_c or sig_p
    any_sig = any_sig or sig
    verdict_rows.append({
        "dose_mult": dose, "alpha_rel": dose * anchor_alpha_rel,
        "early_rate": e["rate"], "late_rate": l["rate"],
        "early_plddt": e["plddt_mean"], "late_plddt": l["plddt_mean"],
        "p_fisher_collapse": p_c, "p_mwu_plddt": p_p,
        "holm_sig_collapse": sig_c, "holm_sig_plddt": sig_p,
        "delta_rate": d_rate, "delta_plddt": d_plddt, "significant": sig,
    })
    print(f"\ndose {dose:g}x  (alpha_rel = {dose * anchor_alpha_rel:.3f})")
    print(f"  collapse   L{LAYER_EARLY} {e['rate']:.1%}  vs  L{LAYER_LATE} {l['rate']:.1%}"
          f"   diff {d_rate:+.1%}   Fisher p = {p_c:.4g}   -> {'SIG' if sig_c else 'ns'}")
    print(f"  mean pLDDT L{LAYER_EARLY} {e['plddt_mean']:.2f} vs  L{LAYER_LATE} {l['plddt_mean']:.2f}"
          f"   diff {d_plddt:+.2f}   MWU    p = {p_p:.4g}   -> {'SIG' if sig_p else 'ns'}")

# --- family 2: did the intervention do ANYTHING vs control? Without this, a layer null is
#     uninterpretable -- two inert conditions look alike for reasons unrelated to layer depth. ---
print()
print("-" * 78)
print("RESPONSIVENESS CHECK -- did steering move this model at all, at either layer?")
print("-" * 78)
ctrl = summary["CONTROL"]
names = [n for n in summary if n != "CONTROL"]
cmps = [compare(summary[n], ctrl) for n in names]
resp_ps = [c[0] for c in cmps] + [c[1] for c in cmps]
resp_hits = holm(resp_ps)
n_c = len(names)

responsive = False
for i, name in enumerate(names):
    p_c, p_p, d_rate, d_plddt = cmps[i]
    hit_c = resp_hits[i] and abs(d_rate) >= MIN_RATE_SHIFT
    hit_p = resp_hits[i + n_c] and abs(d_plddt) >= MIN_PLDDT_SHIFT
    hit = hit_c or hit_p
    responsive = responsive or hit
    flag = "MOVED" if hit else "inert"
    print(f"  {name:26s} collapse {d_rate:+.1%} p={p_c:.4g}   "
          f"pLDDT {d_plddt:+.2f} p={p_p:.4g}   {flag}")
print(f"  (Holm-corrected over {len([p for p in resp_ps if not math.isnan(p)])} tests, "
      f"same effect floor)")

print()
if not responsive:
    print("  ==> THE MODEL IS INERT AT THESE DOSES. No condition differs from control on either")
    print("      readout, so the early-vs-late comparison is comparing two non-effects and says")
    print("      NOTHING about layer depth. Do NOT log this as 'no layer effect' -- log it as")
    print("      'dose too low to test the layer question on this model'. The fix is a higher")
    print("      dose ladder, not a bigger N.")
elif any_sig:
    print("  ==> A LAYER DIFFERENCE SURVIVES MATCHING, at a dose where the model demonstrably")
    print("      responds. This is the first properly-controlled evidence for a layer effect")
    print("      anywhere in the project. Report it for THIS MODEL only -- do not generalize")
    print("      across models, which is the error the original claim made.")
else:
    print("  ==> NO LAYER DIFFERENCE, and this null IS interpretable: the model demonstrably")
    print("      responded to steering at these doses, and the two layers still behaved alike.")
    print("      Layer depth does not modulate steering damage once relative push is matched.")
    print("      This extends §1i-LAYER-REPAIRED from p-IgGen to a second model.")


EARLY vs LATE LAYER at genuinely matched relative push -- Mistral-Prot-134M
anchor_alpha_rel = 0.2000   (1x dose)
early layer = 2, late layer = 7, of 8 total

Condition                    collapsed     rate               95% CI     pLDDT
------------------------------------------------------------------------------
CONTROL                       47/50      94.0%       [83.8%, 97.9%]     42.19
L2_D1x                        35/50      70.0%       [56.2%, 80.9%]     52.15
L2_D2x                        17/50      34.0%       [22.4%, 47.8%]     62.41
L2_D4x                         0/50       0.0%         [0.0%, 7.1%]     72.76
L7_D1x                        39/50      78.0%       [64.8%, 87.2%]     51.01
L7_D2x                        37/50      74.0%       [60.4%, 84.1%]     53.89
L7_D4x                        30/50      60.0%       [46.2%, 72.4%]     59.10

What the ORIGINAL absolute-norm run actually pushed (the confound being repaired):
  layer 2 '1x' was really alpha_rel = 584.0/57082.49 

In [11]:
# --- Persist everything. Nothing enters the paper that is not in one of these files. ---

rows = []
for name, recs in conditions.items():
    if name == "CONTROL":
        layer, mult = None, 0.0
    else:
        layer = LAYER_EARLY if name.startswith(f"L{LAYER_EARLY}_") else LAYER_LATE
        mult = float(name.split("_D")[1].rstrip("x"))
    a_rel = 0.0 if layer is None else mult * anchor_alpha_rel
    for i, r in enumerate(recs):
        rows.append({
            "model": "RaphaelMourad/Mistral-Prot-v1-134M", "condition": name, "idx": i,
            "layer": layer, "dose_mult": mult, "alpha_rel": a_rel,
            "sequence": r["sequence"], "gen_length": len(r["sequence"]),
            "usable_length": sum(1 for a in r["sequence"] if a in VALID_AA),
            "entropy": r["entropy"], "plddt": r["plddt"], "ptm": r["ptm"],
            "fold_ok": r["fold_ok"], "collapse": r["collapse"],
        })
pd.DataFrame(rows).to_csv("mistralprot_layer_matched_sequences.csv", index=False)

pd.DataFrame([{
    "model": "RaphaelMourad/Mistral-Prot-v1-134M", "condition": n, "collapsed": s["k"], "n": s["n"],
    "collapse_rate": s["rate"], "ci_lo": s["ci"][0], "ci_hi": s["ci"][1],
    "fold_ok": s["fold_ok"], "mean_plddt": s["plddt_mean"], "mean_entropy": s["entropy"],
} for n, s in summary.items()]).to_csv("mistralprot_layer_matched_summary.csv", index=False)

pd.DataFrame(verdict_rows).to_csv("mistralprot_layer_matched_layer_tests.csv", index=False)

pd.DataFrame([{
    "model": "RaphaelMourad/Mistral-Prot-v1-134M",
    "anchor_alpha_rel": anchor_alpha_rel, "h_protgpt2_l12": h_protgpt2_l12,
    "n_layers": N_LAYERS, "layer_early": LAYER_EARLY, "layer_late": LAYER_LATE,
    "h_early": h_model[LAYER_EARLY], "h_late": h_model[LAYER_LATE],
    "raw_v_norm_early": raw_norms[LAYER_EARLY], "raw_v_norm_late": raw_norms[LAYER_LATE],
    "matched_norm_early": matched_norms[LAYER_EARLY], "matched_norm_late": matched_norms[LAYER_LATE],
    "old_alpha_rel_early": REFERENCE_NORM / h_model[LAYER_EARLY],
    "old_alpha_rel_late": REFERENCE_NORM / h_model[LAYER_LATE],
    "reference_norm_used_by_original": REFERENCE_NORM,
    "used_uniprot_fallback": USED_FALLBACK,
    "pool_collapse_rate": pool_collapse, "pool_mean_plddt": pool_plddt,
    "utility_gap": util_gap, "repetition_separation": rep_gap,
}]).to_csv("mistralprot_layer_matched_calibration.csv", index=False)

print("Saved:")
for f in ["mistralprot_layer_matched_sequences.csv", "mistralprot_layer_matched_summary.csv",
          "mistralprot_layer_matched_layer_tests.csv", "mistralprot_layer_matched_calibration.csv"]:
    print("  " + f)
print()
print("Download all four into analysis/ and log the verdict in notes/locked-results.md.")
print("The calibration file is the one that proves the push was actually matched -- keep it.")


Saved:
  mistralprot_layer_matched_sequences.csv
  mistralprot_layer_matched_summary.csv
  mistralprot_layer_matched_layer_tests.csv
  mistralprot_layer_matched_calibration.csv

Download all four into analysis/ and log the verdict in notes/locked-results.md.
The calibration file is the one that proves the push was actually matched -- keep it.
